# Can a model remain reliable when future climate doesn't align with historical climate?

**The Core Problem:**  
Standard ML assumes that training and test data come from the *same* distribution. However, climate change violates this assumption. A model trained on past weather patterns may fail catastrophically when faced with a warmer, more volatile future.

Wasserstein Distributionally Robust Optimization (W-DRO) can "immunize" a model against such distribution shifts. Instead of just fitting historical data, W-DRO explicitly trains the model to perform well on the *worst-case* climate distribution within a predefined Wasserstein radius $\epsilon$.

In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import urllib.request
import zipfile
import os

from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.metrics import accuracy_score, average_precision_score

import random
from tqdm import tqdm

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

In [4]:
csv_file = "jena_climate_2009_2016.csv"
zip_file = "jena_climate.zip"
url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

if os.path.exists(csv_file):
  print("Loading directly...")
  df = pd.read_csv(csv_file)
else:
  print("Downloading...")
  urllib.request.urlretrieve(url, zip_file)
  print("Extracting...")
  with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(".")
  os.remove(zip_file)
  print("Loading...")
  df = pd.read_csv(csv_file)

print(f"Loaded {len(df)} rows and {len(df.columns)} features")

Downloading...
Extracting...
Loading...
Loaded 420551 rows and 15 features


In [5]:
# df.info()

df['Date Time'] = pd.to_datetime(df['Date Time'], dayfirst=True)
df.set_index('Date Time', inplace=True)

df_daily = df.resample('D').mean().dropna()
print(f"Resampled to {len(df_daily)} daily records")

Resampled to 2921 daily records


In [6]:
for col in ['wv (m/s)', 'max. wv (m/s)']:
    if col in df_daily.columns:
        df_daily[col] = df_daily[col].replace(-9999.0, np.nan)
        df_daily[col] = df_daily[col].interpolate(method='linear', limit_direction='both')
        df_daily[col] = df_daily[col].fillna(method='ffill').fillna(method='bfill')

/tmp/ipykernel_25716/3726048725.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_daily[col] = df_daily[col].fillna(method='ffill').fillna(method='bfill')


In [7]:
split_i = int(0.6 * len(df_daily))

train_temps = df_daily['T (degC)'].iloc[:split_i]
future_temps = df_daily['T (degC)'].iloc[split_i:]

print(f"Training avg temp (2009-2012): {train_temps.mean():.2f} deg C")
print(f"Future avg temp (2013-2016): {future_temps.mean():.2f} deg C")
print(f"Shift: {future_temps.mean() - train_temps.mean():.2f} deg C")

Training avg temp (2009-2012): 8.99 deg C
Future avg temp (2013-2016): 10.11 deg C
Shift: 1.11 deg C


In [8]:
# 4 physically distinct (independent) features (Temperature, Pressure, Humidity, Wind)
features = ['T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)']

# data = today's weather, tomorrow_temp = next day's temperature
data = df_daily[features].values[:-1].astype(np.float32)
tomorrow_temp = df_daily['T (degC)'].values[1:].astype(np.float32)

In [9]:
split_i = int(0.6*len(data))
train_temps = tomorrow_temp[:split_i]
threshold = np.percentile(train_temps, 90)
print(f"Heatwave threshold (90th %ile of training temps): {threshold:.2f} deg celsius")

Heatwave threshold (90th %ile of training temps): 19.09 deg celsius


In [10]:
labels = (tomorrow_temp > threshold).astype(np.float32).reshape(-1, 1)

In [11]:
mean = data[:split_i].mean(axis=0)
std = data[:split_i].std(axis=0)
data = (data - mean) / std

SEQ_LENGTH = 30
PRED_HORIZON = 7

In [12]:
def create_sequences(data, labels, seq_len, pred_horizon=1):
  Xs, Ys = [], []
  for i in range(len(data) - seq_len - pred_horizon + 1):
    Xs.append(data[i:i+seq_len])
    Ys.append(labels[i+seq_len + pred_horizon - 1])
  return np.array(Xs), np.array(Ys)

In [13]:
X_seq, Y_seq = create_sequences(data, labels, SEQ_LENGTH, pred_horizon=PRED_HORIZON)

 **hindcasting**- treat recent historical data as our "future" test set.

**The Data Split:**  
Since this is a time-series dataset (2009–2016), we keep the chronological order:
- **Training Set (60%):** 2009–2012 (the past we learn from).
- **Validation Set (20%):** 2013–2014 (to tune epsilon).
- **Test Set (20%):** 2015–2016 (the "future" we predict).

In [14]:
train_end = int(0.6 * len(X_seq))
val_end = int(0.8 * len(X_seq))

X_train = X_seq[:train_end]
Y_train = Y_seq[:train_end]
X_val = X_seq[train_end:val_end]
Y_val = Y_seq[train_end:val_end]
X_test = X_seq[val_end:]
Y_test = Y_seq[val_end:]

print(f"Training sequences (2009-2012): {X_train.shape} (Heatwave rate: {Y_train.mean():.3f})")
print(f"Validation sequences (2013-2014): {X_val.shape} (Heatwave rate: {Y_val.mean():.3f})")
print(f"Test sequences (2015-2016): {X_test.shape} (Heatwave rate: {Y_test.mean():.3f})")

Training sequences (2009-2012): (1730, 30, 4) (Heatwave rate: 0.102)
Validation sequences (2013-2014): (577, 30, 4) (Heatwave rate: 0.066)
Test sequences (2015-2016): (577, 30, 4) (Heatwave rate: 0.175)


In [15]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
Y_val_t = torch.tensor(Y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

In [16]:
class HeatwaveClassifier(nn.Module):
  def __init__(self, input_dim=4, hidden=32, num_layers=1):
    super().__init__()
    self.lstm = nn.LSTM(input_dim, hidden, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden, 1)

  def forward(self, x):
    out, _ = self.lstm(x)
    out = self.fc(out[:, -1, :]) # shaped as (batch, days, features) so we grab the last day
    return out

**The adversary tries to find worst-case weather sequences within a physical limit ($\epsilon$), while the model learns to perform well on these adversarial sequences; $\lambda$ auto-adjusts to enforce the distance constraint.**

In [17]:
def train_lstm(model, use_wdro=True, epsilon=1.5, epochs=150, batch_size=128, pgd_steps=10):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()

    # price of transport of prob mass
    lam = torch.tensor(0.1, requires_grad=False)  # dual variable
    loss_his = []

    dataset = TensorDataset(X_train_t, Y_train_t)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in tqdm(range(epochs), desc="Training"):
        epoch_loss = 0.0
        batch_count = 0

        for X_batch, Y_batch in dataloader:
            if not use_wdro:
                logits = model(X_batch)
                loss = loss_fn(logits, Y_batch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
                batch_count += 1
            else:
                # W-DRO: the adversarial attack
                # historical weather
                Z_adv = X_batch.clone().requires_grad_(True)

                # to find worst plausible weather (projection gradient)
                for _ in range(pgd_steps):
                    logits = model(Z_adv)
                    loss = loss_fn(logits, Y_batch)

                    # how far we moved from the initial weather
                    cost = torch.norm(Z_adv - X_batch, dim=(1, 2)).mean()
                    # maximize loss with a penalty on how far we move away
                    obj = loss - lam * cost

                    # ascend on the weather input
                    grad_z = torch.autograd.grad(obj, Z_adv, retain_graph=True)[0]
                    Z_adv = Z_adv + 0.02 * torch.sign(grad_z)

                    Z_adv = torch.clamp(Z_adv, -2, 2)
                    Z_adv = Z_adv.detach().requires_grad_(True)

                Z_adv_final = Z_adv.detach().requires_grad_(True)
                final_logits = model(Z_adv_final)
                final_loss = loss_fn(final_logits, Y_batch)
                final_cost = torch.norm(Z_adv_final - X_batch, dim=(1, 2)).mean()

                # minimize worst-case loss
                optimizer.zero_grad()
                final_loss.backward()
                optimizer.step()

                # update lambda (dual)
                lam = torch.clamp(lam + 0.01 * (final_cost - epsilon), min=0)
                epoch_loss += final_loss.item()
                batch_count += 1

        loss_his.append(epoch_loss / batch_count)

    return loss_his

In [18]:
def recall_at_k(y_true, probs, k_percent=10):
    sorted_idx = np.argsort(probs)[::-1]
    k = int(len(y_true) * k_percent / 100)
    top_k_idx = sorted_idx[:k]
    return y_true[top_k_idx].mean()

def bootstrap_ci(y_true, probs, metric_fn, n_iter=500, ci=95):
    scores = []
    for _ in range(n_iter):
        i = resample(range(len(y_true)), n_samples=len(y_true), replace=True)
        scores.append(metric_fn(y_true[i], probs[i]))
    lower = (100 - ci) / 2
    upper = 100 - lower
    return np.percentile(scores, [lower, upper])

def evaluate(model, X_data, Y_data):
  model.eval()
  with torch.no_grad():
    logits = model(X_data)
    probs = torch.sigmoid(logits).numpy().flatten()
    preds = (probs > 0.5).astype(int)

  y_true = Y_data.numpy().flatten()

  return {
        'y_true': y_true,
        'probs': probs,
        'preds': preds,
        'accuracy': accuracy_score(y_true, preds),
        'brier': brier_score_loss(y_true, probs),
        'auroc': roc_auc_score(y_true, probs),
        'auprc': average_precision_score(y_true, probs),
        'recall10': recall_at_k(y_true, probs, k_percent=10)
    }

In [ ]:
# can be further hypertuned
# print("\n" + "="*70)
# print(" VALIDATING EPSILON ON VALIDATION SET (2013-2014)")
# print("="*70)

# BEST_EPSILON = 0.65

# set_seed()
# model_val = HeatwaveClassifier()
# train_lstm(model_val, use_wdro=True, epsilon=BEST_EPSILON, epochs=30, batch_size=128, pgd_steps=10)

# val_results = evaluate(model_val, X_val_t, Y_val_t)
# print(f"eps = {BEST_EPSILON:.2f} -> Val AUROC: {val_results['auroc']:.4f}, Val AUPRC: {val_results['auprc']:.4f}")
# print(f"Using eps = {BEST_EPSILON} for the final experiment.")

In [19]:
def run_experiment(seed, epsilon=0.65):
    set_seed(seed)

    print("\n" + "="*60)
    print(f"Training Standard LSTM (ERM) - Seed {seed}...")
    print("="*60)

    model_erm = HeatwaveClassifier()
    erm_losses = train_lstm(model_erm, use_wdro=False)

    print("\n" + "="*60)
    print(f"Training W-DRO LSTM - Seed {seed}...")
    print("="*60)

    model_wdro = HeatwaveClassifier()
    wdro_losses = train_lstm(model_wdro, use_wdro=True, epsilon=epsilon, pgd_steps=10)

    erm = evaluate(model_erm, X_test_t, Y_test_t)
    wdro = evaluate(model_wdro, X_test_t, Y_test_t)

    return erm, wdro, erm_losses, wdro_losses

# Run with seed 42
SEED = 42
EPSILON = 0.65

erm, wdro, erm_losses, wdro_losses = run_experiment(SEED, epsilon=EPSILON)


Training Standard LSTM (ERM) - Seed 42...


Training: 100%|██████████| 150/150 [00:29<00:00,  5.10it/s]



Training W-DRO LSTM - Seed 42...


Training: 100%|██████████| 150/150 [11:38<00:00,  4.66s/it]


In [21]:
print("\n" + "="*70)
print(f" PERFORMANCE ON FUTURE WEATHER (2015-2016) - LSTM")
print("="*70)
print(f"{'Metric':<20} | {'ERM LSTM':<15} | {'W-DRO LSTM':<15} | {'Better':<15}")
print("-"*70)

metrics = [
    ('Accuracy', erm['accuracy'], wdro['accuracy'], lambda e, w: 'W-DRO' if w > e else 'ERM'),
    ('Brier Score', erm['brier'], wdro['brier'], lambda e, w: 'W-DRO' if w < e else 'ERM'),
    ('AUROC', erm['auroc'], wdro['auroc'], lambda e, w: 'W-DRO' if w > e else 'ERM'),
    ('AUPRC', erm['auprc'], wdro['auprc'], lambda e, w: 'W-DRO' if w > e else 'ERM'),
    ('Recall@10%', erm['recall10'], wdro['recall10'], lambda e, w: 'W-DRO' if w > e else 'ERM')
]

for name, erm_val, wdro_val, winner_fn in metrics:
    print(f"{name:<20} | {erm_val:.4f}        | {wdro_val:.4f}        | {winner_fn(erm_val, wdro_val):<15}")


 PERFORMANCE ON FUTURE WEATHER (2015-2016) - LSTM
Metric               | ERM LSTM        | W-DRO LSTM      | Better         
----------------------------------------------------------------------
Accuracy             | 0.8163        | 0.8232        | W-DRO          
Brier Score          | 0.1442        | 0.1364        | W-DRO          
AUROC                | 0.8514        | 0.8557        | W-DRO          
AUPRC                | 0.4256        | 0.4405        | W-DRO          
Recall@10%           | 0.4386        | 0.4561        | W-DRO          


In [23]:
print("\n" + "="*70)
print(" 95% CONFIDENCE INTERVALS (Bootstrap, n=500)")
print("="*70)

def recall_metric(y_true, probs, k_percent=10):
    return recall_at_k(y_true, probs, k_percent=k_percent)

def accuracy_from_probs(y_true, probs):
    preds = (probs > 0.5).astype(int)
    return accuracy_score(y_true, preds)

for name, metric_fn in [
    ('Accuracy', accuracy_from_probs),
    ('Brier', brier_score_loss),
    ('AUROC', roc_auc_score),
    ('AUPRC', average_precision_score),
    ('Recall@10%', recall_metric)
]:
    ci_erm = bootstrap_ci(erm['y_true'], erm['probs'], metric_fn)
    ci_wdro = bootstrap_ci(wdro['y_true'], wdro['probs'], metric_fn)

    overlap = not (ci_erm[1] < ci_wdro[0] or ci_wdro[1] < ci_erm[0])
    significance = "Overlap" if overlap else "Statistically Significant"

    print(f"\n{name}:")
    print(f"  ERM:   [{ci_erm[0]:.4f}, {ci_erm[1]:.4f}]")
    print(f"  W-DRO: [{ci_wdro[0]:.4f}, {ci_wdro[1]:.4f}]")
    print(f"  {significance}")


 95% CONFIDENCE INTERVALS (Bootstrap, n=500)

Accuracy:
  ERM:   [0.7859, 0.8440]
  W-DRO: [0.7920, 0.8562]
  Overlap

Brier:
  ERM:   [0.1192, 0.1707]
  W-DRO: [0.1141, 0.1600]
  Overlap

AUROC:
  ERM:   [0.8173, 0.8790]
  W-DRO: [0.8275, 0.8837]
  Overlap

AUPRC:
  ERM:   [0.3524, 0.5231]
  W-DRO: [0.3625, 0.5535]
  Overlap

Recall@10%:
  ERM:   [0.3333, 0.5789]
  W-DRO: [0.3333, 0.5965]
  Overlap


### So, did it actually work?

W-DRO improved across every metric - accuracy, calibration, AUROC, AUPRC, and Recall@10%.
- It caught more of the worst days.

The confidence intervals overlap. That's expected with a single seed.
-  The pattern is clear though - W-DRO's upper bounds are consistently higher across all metrics. More runs would plausibly make W-DRO signficantly different (without overlapping).

---

### Why we care about these metrics

Accuracy is misleading when heatwaves are rare. On a 17.5% heatwave test set, predicting "no heatwave" every day gives 82.5% accuracy - which is useless.

What matters:

- Recall@10% - catching the worst days.
- AUPRC - reliable warnings, not false alarms.
- Brier Score - probabilities you can actually trust. (calibration)

W-DRO improves all of them. (in this run! might slightly differ if we run multiple seeds)


---

>W-DRO doesn't make the model perfect. It makes it somewhat reliable - even when the climate shifts.